## Week 2: Vector Search — Part 3: Using `sqlitesearch` as persistent knowledge base (instead of `minsearch`)

* Replaces nearest neighbour (NN) search with Approximate nearest neighbour (ANN) search to reduce search latency 

```text
NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5
```

### Setup: load model and data

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_faq_data()
index = build_index(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from tqdm.auto import tqdm
import numpy as np

texts = [doc["question"] + " " + doc["answer"] for doc in documents]

vectors = []
batch_size = 50

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Embeddings matrix shape: {X.shape}")

  0%|          | 0/28 [00:00<?, ?it/s]

Embeddings matrix shape: (1368, 384)


### Build the sqlitesearch index

sqlitesearch supports three ANN modes:

- `lsh` (default): up to 100K vectors, random hyperplane projections
- `ivf`: 10K-500K vectors, K-means clustering
- `hnsw`: 10K-1M+ vectors, proximity graph (highest recall)

For our small dataset, `lsh` is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity
reranking.

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"], mode="ivf", db_path="faq_vectors2.db"
)

vs_index.fit(vectors, documents)

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase(index=vs_index, llm_client=openai_client)
assistant.rag("when does the course start")


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



'You can start whenever you want. The videos and GitHub materials are available, and deadlines are listed in the course management platform.'

In [7]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector, num_results=num_results, filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still being accepted.'